# Quantum Kernels and the Cross-Section of Stock Returns: Anatomy of a Vanishing Advantage
**ArXivist-generated reproduction notebook**

Paper: [arXiv:2607.20168](https://arxiv.org/abs/2607.20168) (Junchi Shen, July 2026)
Generated: 2026-07-23

Walks through the paper's kernel-swap control (fidelity quantum kernel vs.
projected quantum kernel vs. classical RBF), closed-form KRR, and the
regularized geometric-difference diagnostic (Eq. 2), on synthetic data --
**the real China A-share dataset is proprietary and not bundled with this
repo** (see `data/README_data.md`).

In [ ]:
import sys, subprocess
sys.path.insert(0, "../src")
result = subprocess.run(["pip", "install", "-e", ".."], capture_output=True, text=True)
print("editable install:", "OK" if result.returncode == 0 else result.stderr[-500:])

import numpy as np
import pandas as pd
print(f"NumPy {np.__version__}, Pandas {pd.__version__}")



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: C:\Users\dhanu\OneDrive\Desktop\Py_pennylane\Scripts\python.exe -m pip install --upgrade pip


Installed kernelspec Py_pennylane in C:\Users\dhanu\AppData\Roaming\jupyter\kernels\py_pennylane
editable install: OK
NumPy 1.26.4, Pandas 2.3.2


## Why this paper matters methodologically

The paper's core finding: **the same quantum kernel, evaluated two different
(both individually defensible-looking) ways, gives opposite conclusions.**
A clean point-in-time universe over 170 windows finds *no* quantum advantage.
A survivorship-screened universe over 60 windows finds the *same* kernel
"dominant." The paper is an anatomy of how that reversal happens -- not a
quantum-computing paper so much as a methodology paper that happens to be
about quantum kernels.

## Component walkthrough: the quantum feature map (Eq. 1)

**This is the highest-risk part of this reproduction.** The paper's Eq. 1
includes a term `H_q` that is never defined:

$$|\psi(x)\rangle = \prod_{r=1}^{R}\left[\prod_q e^{-i\tilde x_q \tilde x_{q+1} Z_q Z_{q+1}} \prod_q e^{-i\tilde x_q Z_q H_q}\right]|0\rangle^{\otimes n}$$

We interpret `H_q` as Hadamard-conjugation of the RZ rotation (the standard
IQP-style single-qubit term). If you have reason to believe a different
convention was intended, change `hq_gate_interpretation` in
`configs/config.yaml` and everything downstream picks it up automatically.

In [8]:
from qkernel_finance.quantum.feature_map import QuantumFeatureMap

fm = QuantumFeatureMap(num_qubits=8, repetitions=2, hq_gate_interpretation="hadamard_conjugated_rz")
x = np.random.uniform(-1, 1, 8)
psi = fm.state(x)
print(f"State shape: {psi.shape}  (expected [256] = 2^8)")
print(f"Norm: {np.abs(psi @ psi.conj()):.6f}  (should be 1.0)")

bloch = fm.bloch_vectors(x)
print(f"Bloch vector shape: {bloch.shape}  (expected [24] =     3 Paulis x 8 qubits)")


AttributeError: module 'numpy' has no attribute 'bool'.
`np.bool` was a deprecated alias for the builtin `bool`. To avoid this error in existing code, use `bool` by itself. Doing this will not modify any behavior and is safe. If you specifically wanted the numpy scalar type, use `np.bool_` here.
The aliases was originally deprecated in NumPy 1.20; for more details and guidance see the original release note at:
    https://numpy.org/devdocs/release/1.20.0-notes.html#deprecations

## The kernel-swap triplet (Sec 4.2-4.3)

Fidelity kernel, projected kernel, and the classical RBF control -- all given
the *same* bandwidth-scaled inputs, so any performance difference isolates
kernel geometry.

In [1]:
from qkernel_finance.features.bandwidth import BandwidthScaler
from qkernel_finance.quantum.kernels import FidelityKernel, ProjectedQuantumKernel
from qkernel_finance.classical.rbf_kernel import ClassicalRBFKernel

scaler = BandwidthScaler()
X = np.random.uniform(-2, 2, size=(12, 8))  # toy: 12 "stocks", 8 top characteristics
X_scaled = scaler.scale(X, lam=0.2)

fidelity = FidelityKernel()
K_fid = fidelity.compute_gram(X_scaled, fm)
print(f"Fidelity Gram matrix: {K_fid.shape}, range [{K_fid.min():.3f}, {K_fid.max():.3f}]  (kernel values in [0,1])")

pqk = ProjectedQuantumKernel()
phi = pqk.bloch_features(X_scaled, fm)
K_proj = pqk.compute_gram(phi, gamma=0.1)
print(f"Projected Gram matrix: {K_proj.shape}, range [{K_proj.min():.3f}, {K_proj.max():.3f}]")

rbf = ClassicalRBFKernel()
K_rbf = rbf.compute_gram(X_scaled, gamma=0.1)
print(f"RBF control Gram matrix: {K_rbf.shape}, range [{K_rbf.min():.3f}, {K_rbf.max():.3f}]")


ModuleNotFoundError: No module named 'pennylane'

## Closed-form kernel ridge regression (Sec 4.3)

$$\hat\alpha = (K + \alpha I)^{-1} y$$

In [ ]:
from qkernel_finance.models.krr import KernelRidgeRegression

krr = KernelRidgeRegression()
y = np.random.normal(0, 0.05, size=12)  # toy 20-day forward returns

alpha_hat = krr.fit(K_fid, y, alpha=0.01)
print(f"Dual coefficients shape: {alpha_hat.shape}  (expected [12])")

# Predict on the same points as a sanity check (should roughly recover y for small alpha)
preds = krr.predict(K_fid, alpha_hat)
print(f"Fit residual (small alpha, in-sample): {np.mean(np.abs(preds - y)):.4f}")


## Regularized geometric difference (Eq. 2, Sec 4.5)

The paper's key mechanism finding: g is always $\gg 1$ (quantum and classical
geometries are genuinely different) but **uncorrelated with realized gains**
($\rho=-0.20$) -- different geometry isn't automatically *better-aligned*
geometry.

In [ ]:
from qkernel_finance.evaluation.geometry import GeometricDifference

geometry = GeometricDifference()
g = geometry.compute(K_rbf, K_fid, lambda_g=1e-6)
print(f"Geometric difference g = {g:.3f}  (paper reports mean 5.4, median 3.4, never below 2.4 across real windows)")


## Full walk-forward pipeline (synthetic smoke test)

Runs `run_study.py`'s logic inline: windowing, top-8 selection, all three
kernel-swap models, and a ridge baseline, over a few synthetic windows.

In [ ]:
import subprocess
result = subprocess.run(
    ["python", "../run_study.py", "--config", "../configs/config.yaml", "--study", "main",
     "--debug", "--output-dir", "/tmp/notebook_demo_results", "--models", "qkrr-fid,krr-rbf,ridge"],
    capture_output=True, text=True,
)
print(result.stdout[-2000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])


In [ ]:
import pandas as pd
for model in ["qkrr-fid", "krr-rbf", "ridge"]:
    path = f"/tmp/notebook_demo_results/{model}_ic_series.csv"
    try:
        series = pd.read_csv(path)["ic"]
        print(f"{model}: mean IC = {series.mean():.4f} over {len(series)} synthetic windows")
    except FileNotFoundError:
        print(f"{model}: no results file (run_study.py may not have completed)")


## Paper's actual reported results (Table 3, Sec 5)

For reference -- **do not expect the synthetic run above to be anywhere
close to these**; they require the real China A-share dataset (Sec 3.1-3.2).

In [ ]:
paper_table3 = {
    "poly2ridge (top-8)": {"mean_ic": 0.0499, "icir": 0.272, "t_stat": 3.55, "hit_rate": 0.629},
    "ridge (top-8)":       {"mean_ic": 0.0494, "icir": 0.247, "t_stat": 3.21, "hit_rate": 0.671},
    "qkrr fidelity":       {"mean_ic": 0.0254, "icir": 0.171, "t_stat": 2.23, "hit_rate": 0.582},
    "krr-rbf control":     {"mean_ic": 0.0208, "icir": 0.161, "t_stat": 2.10, "hit_rate": 0.582},
    "qkrr projected":      {"mean_ic": 0.0168, "icir": 0.134, "t_stat": 1.74, "hit_rate": 0.571},
}
for model, m in paper_table3.items():
    print(f"{model:22s} mean_ic={m['mean_ic']:.4f}  ICIR={m['icir']:.3f}  t={m['t_stat']:.2f}  hit_rate={m['hit_rate']:.3f}")

print("\nKey finding: fidelity QKRR vs RBF control, delta_IC=+0.005, p=0.42 -- statistically indistinguishable.")
print("After Holm correction, NO pairwise difference among 11 models survives (smallest adjusted p=0.43).")


## What to do next

1. **Source real data**: see `data/README_data.md` -- this paper's dataset is
   proprietary (author states "available on request"), the largest
   reproducibility barrier here.
2. **Resolve the H_q ambiguity**: if you can determine the exact intended
   gate from context/correspondence with the author, update
   `configs/config.yaml`'s `quantum.hq_gate_interpretation`.
3. **Full study**: `python run_study.py --study main` (once real data is wired in).
4. **Compare models**: `python compare_models.py --results-dir results/`
5. **Bandwidth/geometry diagnostic**: `python run_bandwidth_geometry_diagnostic.py` (Sec 8, Fig. 2).